# 🌸 Iris Classification with Deep Learning (MLP)
### ML1 Classical Problem → Solved with a Neural Network from Scratch

---

**Problem:** Given 4 flower measurements *(sepal length, sepal width, petal length, petal width)*, classify into one of 3 species: **Setosa, Versicolor, Virginica**.

**In ML1**, we solved this with Logistic Regression / KNN / SVM.  
**Here**, we use a **Multi-Layer Perceptron (MLP)** — a fully connected neural network.

### Why Deep Learning?
| ML1 Method | Limitation | MLP Advantage |
|---|---|---|
| Logistic Regression | Only linear boundaries | Learns non-linear boundaries |
| KNN | No learning, slow inference | Fast inference after training |
| SVM | Hand-picked kernel | Learns its own feature representation |

### Architecture
```
Input (4) → Dense(64, ReLU) → Dense(32, ReLU) → Output(3, Softmax)
```

## 📦 Step 1: Imports

In [51]:
import numpy as np
import csv
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
print("✅ Libraries loaded")

✅ Libraries loaded


## 📂 Step 2: Load Data

In [52]:
# Load the CSV (semicolon-separated)
rows = []
with open('IRIS.csv', 'r') as f:
    reader = csv.reader(f, delimiter=';')
    next(reader)  # skip header
    for row in reader:
        rows.append(row)


# Separate features and labels
X_raw = np.array([[float(r[i]) for i in range(4)] for r in rows])
y_raw = np.array([r[4] for r in rows])

# Encode string labels → integers: setosa=0, versicolor=1, virginica=2
species    = sorted(set(y_raw))
label_map  = {s: i for i, s in enumerate(species)}
y          = np.array([label_map[s] for s in y_raw])

N, D = X_raw.shape
C    = len(species)

print(f"Samples: {N} | Features: {D} | Classes: {C}")
print(f"Label mapping: {label_map}")
for s in species:
    print(f"  {s}: {(y_raw == s).sum()} samples")

Samples: 142 | Features: 4 | Classes: 3
Label mapping: {np.str_('setosa'): 0, np.str_('versicolor'): 1, np.str_('virginica'): 2}
  setosa: 42 samples
  versicolor: 50 samples
  virginica: 50 samples


## ⚙️ Step 3: Preprocessing

### Why StandardScaler?
Neural networks compute **weighted sums** `z = Σ wᵢxᵢ + b`.  
If features have different scales, gradients are unbalanced → slow training.

**Formula:** `x' = (x − μ) / σ` → all features have mean=0, std=1

> ⚠️ **Key rule:** Fit `μ` and `σ` on **train data only**, then apply to val/test.  
> Fitting on test = data leakage = overestimated accuracy.

### Train / Validation / Test Split (70 / 15 / 15)
| Split | Purpose |
|---|---|
| **Train** | Model learns weights from this |
| **Validation** | Monitor overfitting during training |
| **Test** | Final unbiased evaluation — never seen during training |

In [53]:
def stratified_split(X, y, test_frac=0.15, seed=42):
    """Split data while keeping class proportions equal in each split."""
    rng = np.random.RandomState(seed)
    train_idx, test_idx = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_test = max(1, int(len(idx) * test_frac))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])
    return np.array(train_idx), np.array(test_idx)

# Split: 70% train, 15% val, 15% test
train_val_idx, test_idx  = stratified_split(X_raw, y, test_frac=0.15, seed=42)
train_idx, val_idx       = stratified_split(X_raw[train_val_idx], y[train_val_idx], test_frac=0.176, seed=0)
train_idx = train_val_idx[train_idx]
val_idx   = train_val_idx[val_idx]

X_train, y_train = X_raw[train_idx], y[train_idx]
X_val,   y_val   = X_raw[val_idx],   y[val_idx]
X_test,  y_test  = X_raw[test_idx],  y[test_idx]

# StandardScaler — fit on TRAIN only!
mu  = X_train.mean(axis=0)
sig = X_train.std(axis=0) + 1e-8
X_train = (X_train - mu) / sig
X_val   = (X_val   - mu) / sig
X_test  = (X_test  - mu) / sig

print(f"Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}")
print(f"Train mean after scaling ≈ {X_train.mean(axis=0).round(3)}")
print(f"Train std  after scaling ≈ {X_train.std(axis=0).round(3)}")

# One-hot encode labels for CrossEntropy loss
def one_hot(y, C):
    oh = np.zeros((len(y), C))
    oh[np.arange(len(y)), y] = 1.0
    return oh

Y_train = one_hot(y_train, C)
Y_val   = one_hot(y_val,   C)
Y_test  = one_hot(y_test,  C)
print("✅ Preprocessing complete")

Train: 102 | Val: 20 | Test: 20
Train mean after scaling ≈ [ 0. -0. -0.  0.]
Train std  after scaling ≈ [1. 1. 1. 1.]
✅ Preprocessing complete


## ⚡ Step 4: Activation Functions

### ReLU (Hidden Layers)
**Formula:** `f(z) = max(0, z)`  
- Adds **non-linearity** — without it, stacking layers = single linear layer  
- Avoids vanishing gradient (derivative = 1 for z > 0)

### Softmax (Output Layer)
**Formula:** `σ(zᵢ) = exp(zᵢ) / Σⱼ exp(zⱼ)`  
- Converts raw scores → **probability distribution** (all sum to 1)  
- Used for multi-class output

In [54]:
def relu(z):
    """max(0, z) — non-linearity for hidden layers"""
    return np.maximum(0, z)

def relu_grad(z):
    """Derivative of ReLU: 1 if z > 0, else 0"""
    return (z > 0).astype(float)

def softmax(z):
    """Convert logits to probabilities. Subtract max for numerical stability."""
    z_shifted = z - z.max(axis=1, keepdims=True)  # prevents overflow
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

print("✅ Activation functions defined")

# Quick test
z_test = np.array([[2.0, 1.0, 0.5]])
probs  = softmax(z_test)
print(f"Softmax test: {probs.round(3)} — sum = {probs.sum():.1f} ✓")

✅ Activation functions defined
Softmax test: [[0.629 0.231 0.14 ]] — sum = 1.0 ✓


## 🏗️ Step 5: Network Architecture & Weight Initialization

### Architecture: `4 → 64 → 32 → 3`

### He Initialization
**Formula:** `W ~ N(0, √(2 / fan_in))`  
- Designed for **ReLU** activations  
- Keeps the variance of activations constant across layers  
- Prevents vanishing/exploding gradients at the start of training

In [55]:
# Layer sizes: [input=4, hidden1=64, hidden2=32, output=3]
layer_dims = [D, 64, 32, C]

def init_params(layer_dims):
    """Initialize weights with He init, biases with zeros."""
    params = {}
    for l in range(1, len(layer_dims)):
        fan_in = layer_dims[l - 1]
        params[f'W{l}'] = np.random.randn(fan_in, layer_dims[l]) * np.sqrt(2.0 / fan_in)
        params[f'b{l}'] = np.zeros((1, layer_dims[l]))
    return params

params = init_params(layer_dims)
total_params = sum(p.size for p in params.values())

print("Network Architecture:")
for l in range(1, len(layer_dims)):
    print(f"  Layer {l}: {layer_dims[l-1]} → {layer_dims[l]}  | W shape: {params[f'W{l}'].shape}")
print(f"\nTotal trainable parameters: {total_params}")

Network Architecture:
  Layer 1: 4 → 64  | W shape: (4, 64)
  Layer 2: 64 → 32  | W shape: (64, 32)
  Layer 3: 32 → 3  | W shape: (32, 3)

Total trainable parameters: 2499


## 🔄 Step 6: Forward Pass & Backpropagation

### Forward Pass
For each layer: `z = W · x + b` then `a = activation(z)`

### Dropout (Regularization)
- During training: randomly zero **30%** of neurons  
- Forces the network not to rely on any single neuron  
- Acts like training an **ensemble** of smaller networks  
- At test time: no dropout (use full network)

### Backpropagation (Chain Rule)
Output layer gradient (Softmax + CrossEntropy gives a clean result):  
`δ_L = ŷ − y_true`

Hidden layer gradients:  
`δ_l = (δ_{l+1} · Wᵀ) ⊙ ReLU'(z_l)`

Weight update gradients:  
`∂L/∂W_l = aᵀ_{l-1} · δ_l / N`

In [56]:
def forward(X, params, dropout_rate=0.3, training=True):
    """Forward pass through all layers. Returns predictions + cache for backprop."""
    cache = {'a0': X}
    L = len(layer_dims) - 1
    a = X

    for l in range(1, L + 1):
        W, b = params[f'W{l}'], params[f'b{l}']
        z = a @ W + b                    # linear: (N, fan_in) × (fan_in, fan_out)
        cache[f'z{l}'] = z

        if l < L:                        # hidden layers → ReLU + Dropout
            a = relu(z)
            if training and dropout_rate > 0:
                mask = (np.random.rand(*a.shape) > dropout_rate).astype(float)
                a = a * mask / (1 - dropout_rate)   # inverted dropout
                cache[f'mask{l}'] = mask
            else:
                cache[f'mask{l}'] = None
        else:                            # output layer → Softmax
            a = softmax(z)

        cache[f'a{l}'] = a

    return a, cache


def backward(cache, Y_true, params, dropout_rate=0.3, l2_lambda=1e-4):
    """Backpropagation: compute gradients for all weights using chain rule."""
    N = Y_true.shape[0]
    L = len(layer_dims) - 1
    grads = {}

    # Output layer: clean gradient from Softmax + CrossEntropy
    delta = (cache[f'a{L}'] - Y_true) / N

    for l in reversed(range(1, L + 1)):
        a_prev = cache[f'a{l-1}']
        W_l    = params[f'W{l}']

        grads[f'W{l}'] = a_prev.T @ delta + l2_lambda * W_l   # + L2 regularization
        grads[f'b{l}'] = delta.sum(axis=0, keepdims=True)

        if l > 1:
            delta = delta @ W_l.T                              # propagate error
            if dropout_rate > 0 and cache.get(f'mask{l-1}') is not None:
                delta = delta * cache[f'mask{l-1}'] / (1 - dropout_rate)
            delta = delta * relu_grad(cache[f'z{l-1}'])       # ReLU gate

    return grads

print("✅ Forward and backward pass defined")

✅ Forward and backward pass defined


## 📉 Step 7: Loss Function & Adam Optimizer

### CrossEntropy Loss
**Formula:** `L = −(1/N) Σ Σ y_true · log(ŷ)`  
- Penalizes **confident wrong predictions** heavily  
- Combined with L2 regularization: `L_total = L_CE + (λ/2) · Σ‖W‖²`

### Adam Optimizer
Better than plain SGD — adapts learning rate per weight:
```
m = β₁·m + (1−β₁)·g        ← momentum (1st moment)
v = β₂·v + (1−β₂)·g²       ← RMSprop  (2nd moment)
θ = θ − lr · m̂ / (√v̂ + ε)  ← update
```

In [57]:
def compute_loss(y_pred, Y_true, params, l2_lambda=1e-4):
    """CrossEntropy loss + L2 regularization."""
    N   = Y_true.shape[0]
    ce  = -np.sum(Y_true * np.log(y_pred + 1e-12)) / N
    l2  = (l2_lambda / 2) * sum(np.sum(params[f'W{l}']**2) for l in range(1, len(layer_dims)))
    return ce + l2, ce


def init_adam(params):
    m = {k: np.zeros_like(v) for k, v in params.items()}
    v = {k: np.zeros_like(v) for k, v in params.items()}
    return m, v


def adam_update(params, grads, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    """Adam: adaptive learning rate optimizer."""
    L = len(layer_dims) - 1
    for l in range(1, L + 1):
        for p in ['W', 'b']:
            key       = f'{p}{l}'
            m[key]    = b1 * m[key] + (1 - b1) * grads[key]
            v[key]    = b2 * v[key] + (1 - b2) * grads[key]**2
            m_hat     = m[key] / (1 - b1**t)   # bias correction
            v_hat     = v[key] / (1 - b2**t)
            params[key] -= lr * m_hat / (np.sqrt(v_hat) + eps)
    return params, m, v

print("✅ Loss function and Adam optimizer defined")

✅ Loss function and Adam optimizer defined


## 🏋️ Step 8: Training Loop

### Mini-Batch Gradient Descent
- Split train data into **batches of 16**  
- Each batch: forward → compute loss → backward → update weights  
- Better gradient estimate than single-sample SGD

### Early Stopping
- Monitor **validation loss** after every epoch  
- If it doesn't improve for 40 epochs → stop training  
- Restore the **best checkpoint** (lowest val loss)  
- Prevents overfitting to training data

In [58]:
EPOCHS     = 500
BATCH_SIZE = 16
LR         = 0.001
DROPOUT    = 0.3
L2         = 1e-4
PATIENCE   = 40

m_adam, v_adam = init_adam(params)
t = 0  # Adam time step

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss  = np.inf
best_params    = None
patience_count = 0

print(f"Training | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR} | Early stop patience: {PATIENCE}\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train Acc':>10} | {'Val Acc':>10}")
print("-" * 56)

for epoch in range(1, EPOCHS + 1):

    # ── Mini-batch training ─────────────────────────────────────────────
    idx = np.random.permutation(len(y_train))
    for start in range(0, len(y_train), BATCH_SIZE):
        b       = idx[start:start + BATCH_SIZE]
        pred_b, cache_b = forward(X_train[b], params, DROPOUT, training=True)
        loss_b, _       = compute_loss(pred_b, Y_train[b], params, L2)
        grads           = backward(cache_b, Y_train[b], params, DROPOUT, L2)
        t += 1
        params, m_adam, v_adam = adam_update(params, grads, m_adam, v_adam, t, LR)

    # ── Evaluate (no dropout) ───────────────────────────────────────────
    tr_pred, _ = forward(X_train, params, dropout_rate=0, training=False)
    vl_pred, _ = forward(X_val,   params, dropout_rate=0, training=False)

    tr_loss, _ = compute_loss(tr_pred, Y_train, params, L2)
    vl_loss, _ = compute_loss(vl_pred, Y_val,   params, L2)
    tr_acc     = (tr_pred.argmax(1) == y_train).mean()
    vl_acc     = (vl_pred.argmax(1) == y_val).mean()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    # ── Early stopping ──────────────────────────────────────────────────
    if vl_loss < best_val_loss - 1e-5:
        best_val_loss = vl_loss
        best_params   = {k: v.copy() for k, v in params.items()}
        patience_count = 0
    else:
        patience_count += 1

    if patience_count >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch}")
        break

    if epoch % 50 == 0 or epoch == 1:
        print(f"{epoch:>6} | {tr_loss:>10.4f} | {vl_loss:>10.4f} | {tr_acc:>10.4f} | {vl_acc:>10.4f}")

params = best_params
print(f"\n✅ Training done. Best val loss: {best_val_loss:.4f}")

Training | Epochs: 500 | Batch: 16 | LR: 0.001 | Early stop patience: 40

 Epoch | Train Loss |   Val Loss |  Train Acc |    Val Acc
--------------------------------------------------------
     1 |     0.6350 |     0.7585 |     0.7843 |     0.7000
    50 |     0.1778 |     0.1408 |     0.9314 |     1.0000
   100 |     0.1011 |     0.0574 |     0.9706 |     1.0000
   150 |     0.0693 |     0.0279 |     0.9804 |     1.0000
   200 |     0.0591 |     0.0221 |     0.9902 |     1.0000
   250 |     0.0484 |     0.0216 |     0.9804 |     1.0000
   300 |     0.0440 |     0.0188 |     0.9804 |     1.0000
   350 |     0.0402 |     0.0181 |     0.9902 |     1.0000
   400 |     0.0388 |     0.0155 |     0.9902 |     1.0000

⏹  Early stopping at epoch 406

✅ Training done. Best val loss: 0.0147


## 🧪 Step 9: Test Set Evaluation

The test set was **never seen** during training or hyperparameter tuning.  
This gives us an honest estimate of real-world performance.

### Metrics
- **Accuracy** = correct predictions / total  
- **Precision** = TP / (TP + FP) — when we predict class C, how often are we right?  
- **Recall** = TP / (TP + FN) — of all true class C samples, how many did we catch?  
- **F1** = harmonic mean of Precision & Recall

In [59]:
# Final test evaluation
test_pred, _ = forward(X_test, params, dropout_rate=0, training=False)
test_cls     = test_pred.argmax(axis=1)
test_acc     = (test_cls == y_test).mean()

print(f"🎯 Test Accuracy: {test_acc * 100:.2f}%\n")
print(f"{'Class':12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 46)

for i, name in enumerate(species):
    tp   = ((test_cls == i) & (y_test == i)).sum()
    fp   = ((test_cls == i) & (y_test != i)).sum()
    fn   = ((test_cls != i) & (y_test == i)).sum()
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    f1   = 2 * prec * rec / (prec + rec + 1e-12)
    sup  = (y_test == i).sum()
    print(f"{name:12} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {sup:>10}")

# Confusion matrix
print("\nConfusion Matrix:")
cm = np.zeros((C, C), dtype=int)
for true, pred in zip(y_test, test_cls):
    cm[true, pred] += 1

header = f"{'':12}" + "".join(f"{s:>12}" for s in species)
print(header)
for i, name in enumerate(species):
    row = f"{name:12}" + "".join(f"{cm[i,j]:>12}" for j in range(C))
    print(row)

🎯 Test Accuracy: 100.00%

Class         Precision     Recall         F1    Support
----------------------------------------------
setosa           1.0000     1.0000     1.0000          6
versicolor       1.0000     1.0000     1.0000          7
virginica        1.0000     1.0000     1.0000          7

Confusion Matrix:
                  setosa  versicolor   virginica
setosa                 6           0           0
versicolor             0           7           0
virginica              0           0           7


## 📊 Step 10: Visualizations

In [60]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Iris MLP — Training Results', fontsize=14, fontweight='bold')

ep = range(1, len(history['train_loss']) + 1)

# ── Loss curves ─────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(ep, history['train_loss'], label='Train Loss', color='steelblue', lw=2)
ax.plot(ep, history['val_loss'],   label='Val Loss',   color='tomato',    lw=2)
ax.set_title('Loss Curves\n(gap = overfitting indicator)')
ax.set_xlabel('Epoch'); ax.set_ylabel('CrossEntropy Loss')
ax.legend(); ax.grid(alpha=0.3)

# ── Accuracy curves ──────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(ep, [a*100 for a in history['train_acc']], label='Train Acc', color='steelblue', lw=2)
ax.plot(ep, [a*100 for a in history['val_acc']],   label='Val Acc',   color='tomato',    lw=2)
ax.axhline(y=test_acc*100, color='green', lw=1.5, ls='--', label=f'Test: {test_acc*100:.1f}%')
ax.set_title('Accuracy Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_ylim([50, 105])
ax.legend(); ax.grid(alpha=0.3)

# ── Confusion matrix ─────────────────────────────────────────────────────────
ax = axes[2]
im = ax.imshow(cm, cmap='Blues')
for i in range(C):
    for j in range(C):
        color = 'white' if cm[i,j] > cm.max()/2 else 'black'
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color=color, fontsize=16, fontweight='bold')
ax.set_xticks(range(C)); ax.set_yticks(range(C))
ax.set_xticklabels([s[:3].capitalize() for s in species])
ax.set_yticklabels([s[:3].capitalize() for s in species])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Test Set)\nDiagonal = correct predictions')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('iris_dl_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plots saved")

✅ Plots saved


C:\Users\Atif\AppData\Local\Temp\ipykernel_8028\2644276038.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ✅ Step 11: Summary

### Full Pipeline Recap

| Step | What | Why |
|---|---|---|
| **StandardScaler** | `x' = (x−μ)/σ` on train only | Balanced gradients, no data leakage |
| **Stratified Split** | 70/15/15 | Preserve class distribution in all splits |
| **He Init** | `W ~ N(0, √2/fan_in)` | Stable gradients at start for ReLU |
| **ReLU** | `max(0, z)` | Non-linearity, no vanishing gradient |
| **Softmax** | `exp(z)/Σexp` | Multi-class probability output |
| **CrossEntropy** | `−Σ y·log(ŷ)` | Penalizes wrong confident predictions |
| **Backprop** | Chain rule layer by layer | Learn weights from error signal |
| **Adam** | Adaptive lr per weight | Faster, more stable than SGD |
| **Dropout 30%** | Zero random neurons | Regularization, prevents overfitting |
| **L2 Reg** | `λ‖W‖²` on loss | Penalizes large weights |
| **Early Stopping** | Restore best checkpoint | Stops when val loss plateaus |

### DL vs ML1

> A **Logistic Regression** model = just the output layer of our MLP (1 layer, softmax).  
> Adding hidden layers with ReLU gives the network the ability to learn **non-linear decision boundaries** — something LR, KNN, and linear SVM cannot do without manual feature engineering.